In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("diabetes_dataset.csv")

In [3]:
df.shape

(5100, 16)

In [5]:
df.head()

,Age,Gender,Glucose,BMI,BloodPressure,Insulin,SkinThickness,DiabetesPedigreeFunction,FrequentUrination,ExcessiveThirst,SuddenWeightLoss,Fatigue,BlurredVision,FamilyHistoryDiabetes,Disease,Medicine
0,72,1,124,29.1,81,25.4,21,1.982,0,0,0,1,0,1,Prediabetes,Diet + Exercise
1,64,1,116,29.5,85,34.1,25,1.180,1,1,0,1,0,1,Prediabetes,Diet + Exercise
2,42,0,121,32.1,85,27.0,30,2.033,1,1,0,0,0,1,Prediabetes,Diet + Exercise
3,67,0,93,24.4,73,6.2,17,1.553,0,0,0,0,0,1,Normal,Healthy lifestyle
4,46,0,74,25.5,83,17.8,16,2.388,0,0,0,0,0,1,Normal,Healthy lifestyle


In [6]:
# Encode categorical target columns (Disease, Medicine)
disease_encoder = LabelEncoder()
medicine_encoder = LabelEncoder()

df["Disease_encoded"] = disease_encoder.fit_transform(df["Disease"])
df["Medicine_encoded"] = medicine_encoder.fit_transform(df["Medicine"])

In [7]:
X = df.drop(["Disease", "Medicine", "Disease_encoded", "Medicine_encoded"], axis=1)
y_disease = df["Disease_encoded"]
y_medicine = df["Medicine_encoded"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_disease, test_size=0.2, random_state=42, stratify=y_disease
)

In [9]:
model = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)

In [10]:
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, n_estimators=150, random_state=42)

In [11]:
y_pred = model.predict(X_test)

In [12]:
accuracy = accuracy_score(y_test, y_pred)

In [13]:
print(f"Disease Prediction Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=disease_encoder.classes_))

Disease Prediction Accuracy: 98.82%

Classification Report:
                      precision    recall  f1-score   support

Gestational Diabetes       0.99      0.99      0.99       170
  High Risk Diabetes       0.94      1.00      0.97       170
              Normal       1.00      1.00      1.00       170
         Prediabetes       0.99      1.00      1.00       170
     Type 1 Diabetes       1.00      1.00      1.00       170
     Type 2 Diabetes       1.00      0.94      0.97       170

            accuracy                           0.99      1020
           macro avg       0.99      0.99      0.99      1020
        weighted avg       0.99      0.99      0.99      1020



In [14]:
# Train a second model to predict the recommended Medicine
X_train_m, X_test_m, ym_train, ym_test = train_test_split(
    X, y_medicine, test_size=0.2, random_state=42, stratify=y_medicine
)

medicine_model = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=42)
medicine_model.fit(X_train_m, ym_train)

ym_pred = medicine_model.predict(X_test_m)
medicine_accuracy = accuracy_score(ym_test, ym_pred)

print(f"Medicine Prediction Accuracy: {medicine_accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(ym_test, ym_pred, target_names=medicine_encoder.classes_))

Medicine Prediction Accuracy: 98.82%

Classification Report:
                                     precision    recall  f1-score   support

                    Diet + Exercise       0.99      1.00      1.00       170
                  Healthy lifestyle       1.00      1.00      1.00       170
                    Insulin Therapy       1.00      1.00      1.00       170
Lifestyle modification + monitoring       0.94      1.00      0.97       170
                          Metformin       1.00      0.94      0.97       170
      Pregnancy diabetes management       0.99      0.99      0.99       170

                           accuracy                           0.99      1020
                          macro avg       0.99      0.99      0.99      1020
                       weighted avg       0.99      0.99      0.99      1020



In [15]:
import joblib

joblib.dump(model, "diabetes_disease_model.pkl")
joblib.dump(medicine_model, "diabetes_medicine_model.pkl")
joblib.dump(disease_encoder, "diabetes_disease_encoder.pkl")
joblib.dump(medicine_encoder, "diabetes_medicine_encoder.pkl")

print("Diabetes Disease & Medicine Models Saved Successfully")

Diabetes Disease & Medicine Models Saved Successfully


In [16]:
# Example: predict disease + medicine for a new patient
sample = X_test.iloc[[0]]
pred_disease = disease_encoder.inverse_transform(model.predict(sample))
pred_medicine = medicine_encoder.inverse_transform(medicine_model.predict(sample))

print("Predicted Disease:", pred_disease[0])
print("Recommended Medicine:", pred_medicine[0])

Predicted Disease: Prediabetes
Recommended Medicine: Diet + Exercise
